## Solution to the Optimization Problem

Given the opponent's simulated feasible squads $\mathbb{W}_o$ (from `copula_estimation.ipynb`),
solve the manager's own "beat the opponent" problem:

$$\max_{w\in\mathbb{W}} \text{Prob}\{w^T\delta > G^{(r')}(\mathbb{W}_o, \delta)\} \tag{Eq:maxProb}$$

where $\delta$ is the (uncertain) vector of player points, $G_o = w_o^T\delta$ is one
opponent squad's realized score, $G^{(r)}$ is the $r$-th order statistic of
$\{G_o\}_{o=1}^O$, $r' = O + 1 - r$, and $r$ trades off risk: 50% (beat the median
opponent squad) is risk-neutral, up to 90% (beat nearly all of them) is risk-seeking.

**Theorem** (mean-variance reduction): with $Y_w = w^T\delta - G^{(r')} \sim
N(\mu_{Y_w}, \sigma^2_{Y_w})$,

$$
\mu_{Y_w} = w^T\mu_\delta - \mu_{G^{(r')}}, \qquad
\sigma^2_{Y_w} = w^T\Sigma_\delta w + \sigma^2_{G^{(r')}} - 2w^T\sigma_{\delta,G^{(r')}}
$$

and the solution to Eq:maxProb is $w(\lambda) \in \arg\max_{w\in\mathbb{W}} \mu_{Y_w} \mp
\lambda\sigma^2_{Y_w}$ ($-\lambda$ if $\mu_{Y_w}\geq0$, $+\lambda$ otherwise) for some
$\lambda\geq0$, with $\mu_{G^{(r')}}, \sigma^2_{G^{(r')}}, \sigma_{\delta,G^{(r')}}$
estimated by Monte Carlo since they don't depend on $w$.

**This notebook's plan**, following the Lemma and Algorithm in the write-up:

1. Generate $\mathbb{W}_o$ (already done in `copula_estimation.ipynb` -- reload its
   pipeline here and rerun for one opponent/gameweek).
2. Build $\mu_\delta$, $\Sigma_\delta$ for the player-points vector $\delta$.
3. Monte Carlo: draw $\delta \sim N(\mu_\delta,\Sigma_\delta)$ many times; for each draw,
   score every $w_o\in\mathbb{W}_o$ and take the $r'$-th order statistic $G^{(r')}$;
   estimate $\mu_{G^{(r')}}, \sigma^2_{G^{(r')}}, \sigma_{\delta,G^{(r')}}$ from these paired
   samples.
4. Solve the mean-variance problem for a grid of $\lambda$ via a Gurobi MIQP over the
   real squad constraints (15 players, 2/5/5/3 by position, budget, transfer limit) --
   this _is_ $\mathbb{W}$, defined by linear/quadratic constraints rather than an
   explicit enumerated candidate list.
5. Pick $\lambda^* = \arg\max_\lambda \widehat{\text{Prob}}(Y_{w_\lambda} > 0)$ from the
   same Monte Carlo sample, and return $w^* = w_{\lambda^*}$.


### Setup

Reloads the position-pool / `alpha_manager` / vine-fitting machinery from
`copula_estimation.ipynb` (copied rather than `%run`, so this notebook doesn't also
re-trigger that notebook's expensive 33-gameweek bulk loop as a side effect) so
`run_copula_pipeline` is available to regenerate $\mathbb{W}_o$ for a chosen
opponent/gameweek with current code.


In [9]:
import ast

import numpy as np
import pandas as pd
import joblib
import pyvinecopulib as pv
from scipy.stats import kendalltau
import gurobipy as gp
from gurobipy import GRB

player_data = pd.read_csv('../../rolled_data_24_25.csv')
league_selections = pd.read_csv('../../league_selections_df.csv')

LEAGUE_FITS = {
    'Goalkeeper': joblib.load('../estimates/league_models_gk'),
    'Defender': joblib.load('../estimates/league_models_def'),
    'Midfielder': joblib.load('../estimates/league_models_mid'),
    'Forward': joblib.load('../estimates/league_models_fwd'),
}

POSITION_BINARY_COL = {
    'Goalkeeper': 'goalkeeper_binary',
    'Defender': 'defender_binary',
    'Midfielder': 'midfielder_binary',
    'Forward': 'forward_binary',
}

SQUAD_SLOTS = {'Goalkeeper': 2, 'Defender': 5, 'Midfielder': 5, 'Forward': 3}
POSITIONS = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
BICOP_FAMILY_SET = [pv.BicopFamily.indep, pv.BicopFamily.gaussian, pv.BicopFamily.clayton,
                     pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.joe]
TRUNC_LVL = 15


def get_league_fit_for_round(league_fits, target_round):
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(f"No league fit available at or before round {target_round}")
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    pool = player_data[(player_data['round'] == rnd) & (player_data['position'] == position)] \
        .sort_values('element')
    elements = pool['element'].to_numpy()
    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw


def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = X_std_new @ beta_m + manager_fit["gamma_mean"] * y_prev
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)


def marginal_inclusion_probabilities(weights, k, rng, n_sim=3000):
    weights = np.clip(np.asarray(weights, dtype=float), 1e-12, None)
    n = len(weights)
    U = rng.uniform(size=(n_sim, n))
    keys = U ** (1.0 / weights)[None, :]
    topk_idx = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
    counts = np.bincount(topk_idx.ravel(), minlength=n)
    return counts / n_sim


def detect_squad_overhaul(team_id, gw, league_selections, threshold=8):
    prev_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw - 1)]
    cur_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw)]
    if len(prev_rows) == 0 or len(cur_rows) == 0:
        return None, None
    prev_squad = set(ast.literal_eval(prev_rows.iloc[0]['squad']))
    cur_squad = set(ast.literal_eval(cur_rows.iloc[0]['squad']))
    n_changed = len(cur_squad - prev_squad)
    return n_changed, n_changed > threshold


def compute_free_transfers_available(team_id, target_round, league_selections, cap=5, overhaul_threshold=8):
    ft_bank = 1
    for w in range(2, target_round):
        n_changed, is_overhaul = detect_squad_overhaul(team_id, w, league_selections, threshold=overhaul_threshold)
        if n_changed is None:
            continue
        if is_overhaul:
            ft_bank = min(cap, ft_bank + 1)
        elif n_changed <= ft_bank:
            ft_bank = min(cap, (ft_bank - n_changed) + 1)
        else:
            ft_bank = 1
    return ft_bank


def get_owned(team_id, position, rnd, elements):
    rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == rnd)]
    if len(rows) == 0:
        return None
    y = np.asarray(ast.literal_eval(rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float)
    if len(y) != len(elements):
        return None
    return dict(zip(elements.tolist(), y.tolist()))


def position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd):
    elements, X_raw = _position_pool(player_data, position, rnd, league_fit['feats'])
    prev_elements, _ = _position_pool(player_data, position, rnd - 1, league_fit['feats'])
    owned_prev_map = get_owned(team_id, position, rnd - 1, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    owned_map = get_owned(team_id, position, rnd, elements)
    return dict(zip(elements.tolist(), alpha.tolist())), owned_map


def position_marginals_at_week(team_id, position, manager_fit, league_fit, rnd, rng):
    alpha_map, owned_map = position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd)
    elements = np.array(list(alpha_map.keys()))
    alpha = np.array(list(alpha_map.values()))
    p = marginal_inclusion_probabilities(alpha, SQUAD_SLOTS[position], rng)
    return dict(zip(elements.tolist(), p.tolist())), owned_map


def latent_uniforms(X_hist, P_hist, rng):
    a = np.where(X_hist.values == 0, 0.0, 1.0 - P_hist.values)
    b = np.where(X_hist.values == 0, 1.0 - P_hist.values, 1.0)
    draws = rng.uniform(size=X_hist.shape)
    U = a + draws * (b - a)
    return pd.DataFrame(U, index=X_hist.index, columns=X_hist.columns)


def run_copula_pipeline(GW, TEAM_ID, N_TOP=30, N_SAMPLES=20_000, T_LIMIT=None, min_history_weeks=5,
                         seed=42, verbose=True):
    """Verbatim from copula_estimation.ipynb -- see that notebook for the full
    derivation/markdown of each step."""
    if T_LIMIT is None:
        T_LIMIT = compute_free_transfers_available(TEAM_ID, GW, league_selections)

    rng = np.random.default_rng(seed)
    manager_fits = joblib.load(f'./estimates/manager_fits_{GW}.joblib')[GW]

    pool_records = []
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, GW, rng)
        ranked = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
        top_elements = {e for e, _ in ranked[:N_TOP]}
        owned_elements = {e for e, v in (owned_map or {}).items() if v == 1}
        for e in top_elements | owned_elements:
            pool_records.append({"element": e, "position": position})
    pool_df = pd.DataFrame(pool_records).sort_values(['position', 'element']).reset_index(drop=True)
    elements_ordered = pool_df['element'].tolist()
    elements_arr = np.array(elements_ordered)
    position_arr = pool_df['position'].to_numpy()
    M = len(elements_ordered)

    all_history_rounds = sorted(
        league_selections.loc[
            (league_selections['team_id'] == TEAM_ID) & (league_selections['round'] < GW) & (league_selections['round'] >= 4),
            'round'
        ].unique().tolist()
    )
    overhaul_rounds = [r for r in all_history_rounds if detect_squad_overhaul(TEAM_ID, r, league_selections)[1]]
    history_rounds = [r for r in all_history_rounds if r not in overhaul_rounds]
    if len(history_rounds) < min_history_weeks:
        if verbose:
            print(f"GW{GW} team {TEAM_ID}: only {len(history_rounds)} non-overhaul history weeks "
                  f"(< {min_history_weeks}), skipping")
        return None

    X_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    P_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        pos_elements = pool_df.loc[pool_df['position'] == position, 'element'].tolist()
        for rnd in history_rounds:
            p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, rnd, rng)
            for e in pos_elements:
                X_hist.loc[rnd, e] = (owned_map or {}).get(e, 0.0)
                P_hist.loc[rnd, e] = p_map.get(e, 1e-6)
    U_vals = latent_uniforms(X_hist, P_hist, rng).values

    tau_matrix = np.zeros((M, M))
    for i in range(M):
        for j in range(i + 1, M):
            tau, _ = kendalltau(U_vals[:, i], U_vals[:, j])
            tau = 0.0 if np.isnan(tau) else tau
            tau_matrix[i, j] = tau_matrix[j, i] = tau
    hub_scores = np.abs(tau_matrix).sum(axis=1)
    cvine_order = (np.argsort(-hub_scores) + 1).tolist()

    cvine_structure = pv.CVineStructure(order=cvine_order, trunc_lvl=TRUNC_LVL)
    cvine_controls = pv.FitControlsVinecop(family_set=BICOP_FAMILY_SET, selection_criterion="aic", trunc_lvl=TRUNC_LVL)
    vine = pv.Vinecop.from_data(U_vals, structure=cvine_structure, controls=cvine_controls)
    aic = vine.aic(U_vals)

    alpha_target, current_owned = {}, {}
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        alpha_map, owned_map = position_alpha_at_week(TEAM_ID, position, manager_fit, league_fit, GW)
        for e in pool_df.loc[pool_df['position'] == position, 'element']:
            alpha_target[e] = alpha_map.get(e, 1e-12)
            current_owned[e] = (owned_map or {}).get(e, 0.0)
    alpha_vec = np.array([alpha_target[e] for e in elements_ordered])
    current_vec = np.array([current_owned[e] for e in elements_ordered])
    value_map = player_data.loc[player_data['round'] == GW].set_index('element')['value'].to_dict()
    value_vec = np.array([value_map.get(e, np.nan) for e in elements_ordered])
    current_squad_value = float(current_vec @ value_vec)

    sim_U = vine.simulate(n=N_SAMPLES, seeds=[seed])
    w_star = np.zeros_like(sim_U, dtype=int)
    for position, k in SQUAD_SLOTS.items():
        idxs = np.where(position_arr == position)[0]
        w = np.clip(alpha_vec[idxs], 1e-12, None)
        keys = sim_U[:, idxs] ** (1.0 / w)[None, :]
        topk = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
        rows = np.repeat(np.arange(sim_U.shape[0]), k)
        cols = idxs[topk.ravel()]
        w_star[rows, cols] = 1

    transfers = 0.5 * np.abs(w_star - current_vec[None, :]).sum(axis=1)
    squad_values = w_star @ value_vec
    transfer_ok = transfers <= T_LIMIT
    budget_ok = squad_values <= current_squad_value
    feasible_mask = transfer_ok & budget_ok

    feasible_squads = [tuple(elements_arr[row.astype(bool)].tolist()) for row in w_star[feasible_mask]]
    squad_counts = {}
    for sq in feasible_squads:
        squad_counts[sq] = squad_counts.get(sq, 0) + 1

    result = {
        "GW": GW, "team_id": TEAM_ID, "M": M, "elements_ordered": elements_ordered,
        "n_history_weeks": len(history_rounds), "overhaul_rounds": overhaul_rounds,
        "vine_aic": aic, "current_squad_value": current_squad_value,
        "n_samples": N_SAMPLES, "T_LIMIT": T_LIMIT,
        "n_feasible": int(feasible_mask.sum()), "n_unique_feasible_squads": len(squad_counts),
        "squad_counts": squad_counts,
    }
    if verbose:
        print(f"GW{GW:>2} team {TEAM_ID:>8}: M={M:>3}, history={len(history_rounds):>2}wks "
              f"({len(overhaul_rounds)} overhaul), T'={T_LIMIT}, AIC={aic:>9.1f}, "
              f"feasible={result['n_feasible']:>5}/{N_SAMPLES} ({result['n_unique_feasible_squads']} unique)")
    return result

### Parameters

`OWN_TEAM_ID` is manager 205 -- "us" throughout this whole project (the round-robin
fixture in `longitudinal.ipynb` always computes `opponents_by_round` _for_ manager 205).
`OPPONENT_TEAM_ID`/`GW` continue the running example from `copula_estimation.ipynb`
(GW38's opponent, team 13603, verified well-calibrated and with the longest available
history). `R_PCT` is the risk level $r$ (fraction of opponent squads to beat) --
50% is risk-neutral per the write-up.


In [10]:
OWN_TEAM_ID = 205
OPPONENT_TEAM_ID = 13603
GW = 38
R_PCT = 0.50  # risk-neutral: beat the median of the opponent's feasible squads

print(f"Manager {OWN_TEAM_ID} vs opponent {OPPONENT_TEAM_ID} at GW{GW}, r={R_PCT:.0%}")

Manager 205 vs opponent 13603 at GW38, r=50%


### Step 1: the opponent's feasible squads $\mathbb{W}_o$

Rerun `run_copula_pipeline` for the opponent -- this is exactly `copula_estimation.ipynb`'s
output, regenerated here with current code (dynamic $T'$, budget constraint, per-position
top-k sampling) rather than reloading a possibly-stale saved result.


In [11]:
opp_result = run_copula_pipeline(GW, OPPONENT_TEAM_ID, N_SAMPLES=100_000, seed=42)

opp_elements = opp_result['elements_ordered']
opp_squad_counts = opp_result['squad_counts']  # {squad_tuple: count}
O = len(opp_squad_counts)
print(f"\n{O} unique feasible opponent squads (Wo) to score against")

GW38 team    13603: M=120, history=30wks (4 overhaul), T'=3, AIC=   -907.0, feasible= 7573/100000 (7246 unique)

7246 unique feasible opponent squads (Wo) to score against


### Step 2: player-points vector $\delta$ -- $\mu_\delta$ and $\Sigma_\delta$

$\mu_\delta$ comes from `FPL Data/predictors`' walk-forward-validated `xP` regressor
(the most recently built, most rigorously backtested points model in the repo -- see
its own notebooks for the model zoo/feature-selection details): each player's predicted
`xP` for gameweek `GW` itself.

$\Sigma_\delta$ is _not_ the covariance of raw historical points (that would mostly
reflect "some players are nailed starters and some aren't", which $\mu_\delta$ already
captures) -- it's the empirical covariance of the model's own **residuals**
($xP_{\text{actual}} - xP_{\text{pred}}$) across all rounds before `GW`, i.e. the
genuine _prediction uncertainty_ structure (including how errors co-move across
players, e.g. two players on the same team missing a game together).

With ~780 players and lots of missing rounds (transfers, injuries, rotation), the raw
pairwise-deletion covariance is badly indefinite (over half its eigenvalues negative,
checked directly). Fixed via the standard nearest-PSD correction on the _correlation_
matrix (eigenvalue-clip then renormalize to unit diagonal, then rescale by the original
variances) rather than on the raw covariance directly, which preserves each player's
own variance exactly instead of distorting it.


In [12]:
def build_mu_sigma_delta(GW, predictors_dir='../../predictors', min_periods=3, min_eig_frac=1e-4):
    frames = []
    for pos in ['gk', 'def', 'mid', 'fwd']:
        df = pd.read_csv(f'{predictors_dir}/hist/{pos}_preds.csv')
        frames.append(df)
    all_preds = pd.concat(frames, ignore_index=True)

    mu_row = all_preds[all_preds['round'] == GW].drop_duplicates('element')
    hist = all_preds[all_preds['round'] < GW].copy()
    hist['resid'] = hist['xP_actual'] - hist['xP_pred']
    pivot = hist.pivot_table(index='round', columns='element', values='resid')

    elements = sorted(set(mu_row['element']) & set(pivot.columns))
    mu_delta = mu_row.set_index('element').loc[elements, 'xP_pred'].to_numpy(dtype=float)

    cov = pivot[elements].cov(min_periods=min_periods).fillna(0.0).to_numpy()
    cov = (cov + cov.T) / 2
    np.fill_diagonal(cov, np.maximum(np.diag(cov), 1e-4))

    std = np.sqrt(np.diag(cov))
    corr = cov / np.outer(std, std)
    np.fill_diagonal(corr, 1.0)

    eigvals, eigvecs = np.linalg.eigh(corr)
    floor = max(min_eig_frac * eigvals.max(), 1e-8)
    corr_psd = eigvecs @ np.diag(np.clip(eigvals, floor, None)) @ eigvecs.T
    d = np.sqrt(np.diag(corr_psd))
    corr_psd = corr_psd / np.outer(d, d)
    np.fill_diagonal(corr_psd, 1.0)
    corr_psd = (corr_psd + corr_psd.T) / 2

    sigma_delta = corr_psd * np.outer(std, std)
    return elements, mu_delta, sigma_delta


delta_elements, mu_delta, Sigma_delta = build_mu_sigma_delta(GW)
print(f"delta dimension P = {len(delta_elements)}")
print(f"min eigenvalue of Sigma_delta: {np.linalg.eigvalsh(Sigma_delta).min():.2e} (should be >= 0)")
print(f"mu_delta range: [{mu_delta.min():.2f}, {mu_delta.max():.2f}], mean {mu_delta.mean():.2f}")

delta dimension P = 781
min eigenvalue of Sigma_delta: 4.50e-05 (should be >= 0)
mu_delta range: [-0.79, 7.58], mean 1.04


### Step 3: own squad, budget, and player universe

Manager 205's _current_ squad is their GW37 squad (before GW38's decision); their
budget is that squad's value at GW38 prices (same convention as the opponent's budget
constraint in `copula_estimation.ipynb`); their transfer limit $T'_{\text{own}}$ is
their own banked free transfers via `compute_free_transfers_available`.

The player universe for the optimizer is every player with both a $\mu_\delta$/
$\Sigma_\delta$ entry (Step 2) _and_ a valid position/price at `GW` -- the ILP's own
constraints (budget, 2/5/5/3, transfer limit) define $\mathbb{W}$ directly, so there's
no need for the top-N pool truncation used for the opponent's copula stage.


In [13]:
own_row_prev = league_selections[
    (league_selections['team_id'] == OWN_TEAM_ID) & (league_selections['round'] == GW - 1)
].iloc[0]
own_current_squad = set(ast.literal_eval(own_row_prev['squad']))

gw_players = player_data[player_data['round'] == GW].drop_duplicates('element').set_index('element')
universe = sorted(set(delta_elements) & set(gw_players.index))
print(f"player universe: {len(universe)} players (out of {len(delta_elements)} with mu/Sigma, "
      f"{len(gw_players)} with GW{GW} price/position)")

own_budget = float(gw_players.loc[list(own_current_squad & set(gw_players.index)), 'value'].sum())
own_T_limit = compute_free_transfers_available(OWN_TEAM_ID, GW, league_selections)
print(f"own budget (GW37 squad value at GW{GW} prices): {own_budget}")
print(f"own free transfers banked entering GW{GW}: {own_T_limit}")

price_vec = gw_players.loc[universe, 'value'].to_numpy(dtype=float)
position_vec = gw_players.loc[universe, 'position'].to_numpy()
owned_vec = np.array([1.0 if e in own_current_squad else 0.0 for e in universe])

# re-index mu_delta/Sigma_delta onto `universe`'s order
delta_index = {e: i for i, e in enumerate(delta_elements)}
u_idx = np.array([delta_index[e] for e in universe])
mu_u = mu_delta[u_idx]
Sigma_u = Sigma_delta[np.ix_(u_idx, u_idx)]
P = len(universe)

player universe: 781 players (out of 781 with mu/Sigma, 784 with GW38 price/position)
own budget (GW37 squad value at GW38 prices): 1050.0
own free transfers banked entering GW38: 4


### Step 4: Monte Carlo samples of $(\delta, G^{(r')})$

Per the Lemma: draw $\delta\sim N(\mu_\delta,\Sigma_\delta)$; for _that same_ draw, score
every $w_o\in\mathbb{W}_o$ (weighted by how often the copula/vine simulation produced
it -- `opp_squad_counts`); take the $r'$-th order statistic across the opponent's
squads. Repeating this gives paired samples of $(\delta, G^{(r')})$, from which
$\mu_{G^{(r')}}$, $\sigma^2_{G^{(r')}}$, and $\sigma_{\delta,G^{(r')}}$ (the $P$-vector of
$\text{Cov}(\delta_p, G^{(r')})$) are estimated empirically -- none of them depend on our
own decision $w$.


In [14]:
N_MC = 3000
rng = np.random.default_rng(0)

# Wo as a (O, P) binary matrix in `universe`'s column order, weighted by squad_counts
opp_element_index = {e: i for i, e in enumerate(universe)}
Wo_rows, Wo_weights = [], []
for squad, count in opp_squad_counts.items():
    row = np.zeros(P)
    valid = True
    for e in squad:
        idx = opp_element_index.get(e)
        if idx is None:
            valid = False
            break
        row[idx] = 1.0
    if valid:
        Wo_rows.append(row)
        Wo_weights.append(count)
Wo = np.array(Wo_rows)
Wo_weights = np.array(Wo_weights, dtype=float)
Wo_weights /= Wo_weights.sum()
print(f"Wo matrix: {Wo.shape} ({Wo.shape[0]} / {O} opponent squads had all their players in `universe`)")

r_index_from_top = int(np.ceil((1 - R_PCT) * Wo.shape[0]))  # r' = O + 1 - r, 1-indexed from the top
r_index_from_top = min(max(r_index_from_top, 1), Wo.shape[0])

delta_samples = rng.multivariate_normal(mu_u, Sigma_u, size=N_MC, method='eigh')
G_all = delta_samples @ Wo.T  # (N_MC, O_valid): each opponent squad's score per MC draw
G_sorted = np.sort(G_all, axis=1)
G_r = G_sorted[:, -r_index_from_top]  # r'-th order statistic (from the top) per draw

mu_G_r = G_r.mean()
var_G_r = G_r.var(ddof=1)
sigma_delta_G_r = ((delta_samples - mu_u) * (G_r - mu_G_r)[:, None]).mean(axis=0)

print(f"mu_G_r' = {mu_G_r:.3f}, sigma^2_G_r' = {var_G_r:.3f}")
print(f"sigma_delta_G_r' range: [{sigma_delta_G_r.min():.3f}, {sigma_delta_G_r.max():.3f}]")

Wo matrix: (7246, 781) (7246 / 7246 opponent squads had all their players in `universe`)
mu_G_r' = 58.710, sigma^2_G_r' = 167.417
sigma_delta_G_r' range: [-26.820, 31.243]


### Step 5: mean-variance MIQP over a grid of $\lambda$

$\mathbb{W}$ is defined directly by the squad constraints (exactly 15 players, 2 GK/5
DEF/5 MID/3 FWD, budget $\leq$ `own_budget`, at most `own_T_limit` new players vs. the
current squad) rather than an enumerated candidate list -- solved as a Gurobi MIQP.
$\mu_{G^{(r')}}$ and $\sigma^2_{G^{(r')}}$ don't depend on $w$, so they drop out of the
$\arg\max$; only $w^T\mu_\delta - \lambda\big(w^T\Sigma_\delta w - 2w^T\sigma_{\delta,G^{(r')}}\big)$
matters for solving $w(\lambda)$ (the sign convention follows the Theorem, and in
practice $\mu_{Y_w}\geq0$ is the relevant branch once $w$ is any reasonably strong
squad).


In [15]:
def build_mean_variance_model(mu_u, Sigma_u, sigma_delta_G_r, price_vec, position_vec, owned_vec,
                               budget, T_limit):
    """
    Builds the Gurobi model, constraints, and the (lambda-independent) linear/quadratic
    expressions once. Building the dense quadratic expression over ~780 players is the
    expensive part (~10s) -- reusing this model and just swapping the objective per
    lambda (see solve_for_lambda) avoids paying that cost once per grid point.
    """
    P = len(mu_u)
    m = gp.Model('mean_variance')
    m.Params.OutputFlag = 0
    w = m.addVars(P, vtype=GRB.BINARY, name='w')

    m.addConstr(gp.quicksum(w[i] for i in range(P)) == 15)
    for pos, k in SQUAD_SLOTS.items():
        idxs = np.where(position_vec == pos)[0]
        m.addConstr(gp.quicksum(w[i] for i in idxs) == k)
    m.addConstr(gp.quicksum(price_vec[i] * w[i] for i in range(P)) <= budget)
    m.addConstr(gp.quicksum(w[i] * (1 - owned_vec[i]) for i in range(P)) <= T_limit)

    lin = gp.quicksum(mu_u[i] * w[i] for i in range(P))
    quad_var = gp.quicksum(Sigma_u[i, j] * w[i] * w[j] for i in range(P) for j in range(P) if Sigma_u[i, j] != 0)
    lin_cov = gp.quicksum(sigma_delta_G_r[i] * w[i] for i in range(P))
    return m, w, lin, quad_var, lin_cov


def solve_for_lambda(m, w, lin, quad_var, lin_cov, P, lam, sense=1):
    """sense=+1 for the mu_Yw >= 0 branch (maximize mean - lam*var), -1 otherwise
    (maximize mean + lam*var). Returns the optimal binary allocation w (len P) or None
    if infeasible. Reuses the model/constraints from build_mean_variance_model."""
    m.setObjective(lin - sense * lam * (quad_var - 2 * lin_cov), GRB.MAXIMIZE)
    m.optimize()
    if m.status != GRB.OPTIMAL:
        return None
    return np.array([w[i].X for i in range(P)])


mu_Yw_check = float(mu_u @ owned_vec - mu_G_r)  # mu_Yw of the manager's *current* squad, just for the branch sign
sense = 1 if mu_Yw_check >= 0 else -1
print(f"current squad's mu_Yw = {mu_Yw_check:.2f} -> using the "
      f"{'mu - lambda*var' if sense == 1 else 'mu + lambda*var'} branch")

gurobi_model, gurobi_w, lin_expr, quad_var_expr, lin_cov_expr = build_mean_variance_model(
    mu_u, Sigma_u, sigma_delta_G_r, price_vec, position_vec, owned_vec, own_budget, own_T_limit
)

current squad's mu_Yw = 8.44 -> using the mu - lambda*var branch


### Step 6: pick $\lambda^*$ and report $w^*$

For each $\lambda$ in a grid, solve for $w(\lambda)$, then use the _same_ Monte Carlo
$(\delta,G^{(r')})$ samples from Step 4 to estimate $\widehat{\text{Prob}}(Y_{w(\lambda)}>0)
= \frac1{N}\sum_i \mathbb{1}[w(\lambda)^T\delta_i > G^{(r')}_i]$ -- the actual quantity
Eq:maxProb wants maximized. $\lambda^*$ is whichever grid point achieves the highest
empirical probability.


In [16]:
lambda_grid = np.concatenate([[0.0], np.geomspace(0.01, 5.0, 15)])

results = []
for lam in lambda_grid:
    w_lam = solve_for_lambda(gurobi_model, gurobi_w, lin_expr, quad_var_expr, lin_cov_expr, P, lam, sense=sense)
    if w_lam is None:
        continue
    Yw_samples = delta_samples @ w_lam - G_r
    prob_beat = float((Yw_samples > 0).mean())
    mean_score = float(mu_u @ w_lam)
    var_score = float(w_lam @ Sigma_u @ w_lam)
    n_transfers = float(((1 - owned_vec) * w_lam).sum())
    results.append({
        "lambda": lam, "prob_beat_opponent": prob_beat, "mean_score": mean_score,
        "var_score": var_score, "n_transfers": n_transfers, "w": w_lam,
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'w'} for r in results])
print(results_df)

best_idx = results_df['prob_beat_opponent'].idxmax()
lambda_star = results_df.loc[best_idx, 'lambda']
w_star = results[best_idx]['w']
squad_star = [universe[i] for i in range(P) if w_star[i] > 0.5]

print(f"\nlambda* = {lambda_star:.4f}, Prob(beat opponent's {R_PCT:.0%}-percentile squad) = "
      f"{results_df.loc[best_idx, 'prob_beat_opponent']:.1%}")
print(f"transfers made: {results_df.loc[best_idx, 'n_transfers']:.0f} (limit was {own_T_limit})")
print(f"squad value: {sum(price_vec[i] for i in range(P) if w_star[i] > 0.5):.0f} (budget was {own_budget:.0f})")
print(f"w* (element ids): {sorted(squad_star)}")

      lambda  prob_beat_opponent  mean_score   var_score  n_transfers
0   0.000000            0.999000   79.289000  198.362974          4.0
1   0.010000            0.999000   79.289000  198.362974          4.0
2   0.015588            0.999000   79.214000  173.502045          4.0
3   0.024298            0.999000   79.214000  173.502045          4.0
4   0.037875            0.999000   79.214000  173.502045          4.0
5   0.059038            0.999000   79.214000  173.502045          4.0
6   0.092028            0.999333   78.791000  173.211803          4.0
7   0.143450            0.999667   78.312000  176.553888          4.0
8   0.223607            0.999667   76.322000  165.483786          4.0
9   0.348553            1.000000   73.652000  180.672317          4.0
10  0.543316            1.000000   73.652000  180.672317          4.0
11  0.846907            1.000000   73.652000  180.672317          4.0
12  1.320138            0.998000   67.770000  176.844722          4.0
13  2.057799        

### Reading the result

`results_df` shows the mean-variance frontier: as $\lambda$ grows, the optimizer trades
expected score for lower variance, and $\widehat{\text{Prob}}(\text{beat opponent})$ traces
out a curve over that frontier rather than moving monotonically with $\lambda$ (a
higher-variance, higher-mean squad isn't automatically worse at clearing a _fixed_
threshold $G^{(r')}$ -- it depends on where $\mu_{Y_w}$ sits relative to 0). $\lambda^*$
is whichever frontier point actually maximizes that probability, which is the quantity
Eq:maxProb asks for -- not the raw mean-score maximizer ($\lambda=0$) or an arbitrarily
conservative one.

Caveats worth being upfront about:

- $\mu_\delta$ and $\Sigma_\delta$ come from `FPL Data/predictors`' backtested model, so
  this only works for gameweeks that pipeline has already scored (GW6-38 of the
  2024-25 season) -- there's no live "predict a genuinely future gameweek" path here
  yet, same limitation the predictor pipeline itself has.
- $\Sigma_\delta$'s correlation structure comes from historical prediction-residual
  co-movement, not a structural assumption (e.g. same-team clean-sheet correlation) --
  it will pick up real patterns to the extent they were present in GW6-37's residuals,
  but isn't guaranteed to generalize much beyond that window.
- The transfer-limit constraint here only counts _this_ week's swap against the banked
  free-transfer count; it doesn't model taking a deliberate point hit for a swap beyond
  $T'_{\text{own}}$, which a real manager might still rationally do if the expected gain
  outweighs the -4.


### Ground truth: how did $w^*$ actually do?

Everything above is decided _before_ GW38 is played, using only $\mu_\delta$/$\Sigma_\delta$
(model predictions/historical uncertainty) and the opponent's _simulated_ squad
ensemble $\mathbb{W}_o$. Since GW38 has already been played in this dataset, we can now
check against what actually happened: the opponent's real GW38 squad (not a simulated
one), scored on both `xP` (FPL's official expected-points stat -- what the model was
trained to predict) and `total_points` (the real points that actually counted,
including bonus/cards/appearance points that `xP` doesn't capture). The manager's
current (pre-decision) squad is included too, to see whether the optimizer's suggested
transfers would actually have been worth making.


In [17]:
def squad_actuals(squad, gw_players):
    """Sum of actual xP and actual total_points for a squad (list of element ids) at
    the already-played gameweek. Missing players (shouldn't happen for a real squad,
    but guards against a stale universe/price-table mismatch) are excluded and flagged."""
    rows = gw_players.reindex(squad)
    missing = rows.index[rows['xP'].isna()].tolist()
    if missing:
        print(f"  WARNING: {len(missing)} player(s) had no GW{GW} row and were excluded: {missing}")
    rows = rows.dropna(subset=['xP'])
    return float(rows['xP'].sum()), float(rows['total_points'].sum())


opp_actual_row = league_selections[
    (league_selections['team_id'] == OPPONENT_TEAM_ID) & (league_selections['round'] == GW)
].iloc[0]
opp_actual_squad = ast.literal_eval(opp_actual_row['squad'])

own_current_xp, own_current_pts = squad_actuals(sorted(own_current_squad), gw_players)
own_optimal_xp, own_optimal_pts = squad_actuals(squad_star, gw_players)
opp_actual_xp, opp_actual_pts = squad_actuals(opp_actual_squad, gw_players)

comparison_df = pd.DataFrame([
    {"squad": f"own current (GW{GW - 1}, pre-decision)", "actual_xP": own_current_xp, "actual_points": own_current_pts},
    {"squad": f"own optimized w* (lambda*={lambda_star:.4f})", "actual_xP": own_optimal_xp, "actual_points": own_optimal_pts},
    {"squad": f"opponent's real GW{GW} squad", "actual_xP": opp_actual_xp, "actual_points": opp_actual_pts},
])
comparison_df

,squad,actual_xP,actual_points
0,"own current (GW37, pre-decision)",73.7,59.0
1,own optimized w* (lambda*=0.3486),78.1,71.0
2,opponent's real GW38 squad,65.5,50.0


In [18]:
def outcome(a, b):
    return "WIN" if a > b else ("LOSS" if a < b else "TIE")


print(f"xP:     current {own_current_xp:.1f}  ->  optimized {own_optimal_xp:.1f}  "
      f"(delta {own_optimal_xp - own_current_xp:+.1f})  vs opponent {opp_actual_xp:.1f}")
print(f"points: current {own_current_pts:.0f}  ->  optimized {own_optimal_pts:.0f}  "
      f"(delta {own_optimal_pts - own_current_pts:+.0f})  vs opponent {opp_actual_pts:.0f}")
print()
print(f"optimized vs opponent, on xP:     {outcome(own_optimal_xp, opp_actual_xp)}")
print(f"optimized vs opponent, on points: {outcome(own_optimal_pts, opp_actual_pts)}")
print(f"current   vs opponent, on xP:     {outcome(own_current_xp, opp_actual_xp)}")
print(f"current   vs opponent, on points: {outcome(own_current_pts, opp_actual_pts)}")

xP:     current 73.7  ->  optimized 78.1  (delta +4.4)  vs opponent 65.5
points: current 59  ->  optimized 71  (delta +12)  vs opponent 50

optimized vs opponent, on xP:     WIN
optimized vs opponent, on points: WIN
current   vs opponent, on xP:     WIN
current   vs opponent, on points: WIN
